# PyTorch调用算子综合实践

本节包含 Softmax 和 Matmul 两道独立开发题。请使用 Ascend C 实现算子，并接入 PyTorch，使 Python 侧能够传入 NPU Tensor、调用自定义算子并取得 NPU Tensor 输出。

每道题均需独立完成算子实现、PyTorch 接口、编译配置和测试脚本。可选择 Pybind11 或 Torch Library 作为接入方式。

## 前置要求

- 掌握 Ascend C Vector、Cube 算子开发及编译运行的基本方法。
- 理解 PyTorch Tensor 的 shape、dtype、device 和连续存储布局。
- 完成 [PyTorch框架下Kernel直调](../02_AscendC_basic/02.07_kernel_pytorch_call.ipynb) 的学习。

本节只提供题目规格、环境要求和文件清单。各文件的全部内容由开发者自行编写。


---

## 环境准备

### 硬件与软件

| 项目 | 要求 |
| --- | --- |
| NPU | Atlas A2 训练/推理系列或 Atlas A3 训练/推理系列产品 |
| 编译目标 | `dav-2201` |
| CANN | 安装 Toolkit 及匹配的驱动、固件；CANN 9.1.0 及以上 |
| Python 环境 | 安装与 CANN 配套的 PyTorch、`torch_npu`，以及 Jupyter 内核 |
| 构建工具 | CMake 3.16 及以上、make、支持 C++17 的 C++ 编译器，以及 CANN 提供的 Ascend C 编译器 |
| Python 开发依赖 | Python 开发头文件；选择 Pybind11 接入时还需安装 `pybind11` |

### 一键初始化

以本 notebook 所在目录为工作目录，执行下面的单元格即可完成：

1. 自动查找并加载 CANN 环境，将环境变量导入当前 Jupyter 进程。
2. 使用当前内核的 Python 解释器安装 `pybind11` 和 `expecttest`。
3. 检查编译工具、Python 开发头文件，以及独立 Python 进程中的 PyTorch / `torch_npu` 导入和 NPU 运算。
4. 创建 `Sources/03.08/softmax/`、`Sources/03.08/matmul/` 及两种接入方式需要的全部空文件。

CANN 安装在其他位置时，可提前设置 `ASCEND_SET_ENV` 为 `set_env.sh` 的完整路径。初始化会设置 `PY_BIN` 和 `PYTHON_BIN` 为当前内核的解释器，后续编译与运行均使用该环境。

重复执行会保留已有文件的全部内容。新建文件为空，代码仍由开发者自行完成；每道题只需填写所选接入方式对应的文件及共用算子文件。


In [ ]:
import os
import re
import shlex
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path

ASC_ARCH = "dav-2201"
ROOT = Path("Sources/03.08").resolve()
PY_BIN = sys.executable

candidates = []
if os.environ.get("ASCEND_SET_ENV"):
    candidates.append(Path(os.environ["ASCEND_SET_ENV"]))
for name in ("ASCEND_TOOLKIT_HOME", "ASCEND_HOME_PATH"):
    if os.environ.get(name):
        candidates.append(Path(os.environ[name]) / "set_env.sh")
candidates.extend([
    Path("/usr/local/Ascend/cann/set_env.sh"),
    Path("/usr/local/Ascend/ascend-toolkit/set_env.sh"),
])
CANN_SET_ENV = next((path for path in candidates if path.is_file()), None)
if CANN_SET_ENV is None:
    raise FileNotFoundError(
        "未找到 CANN set_env.sh，请先安装 CANN Toolkit 或设置 ASCEND_SET_ENV。"
    )

result = subprocess.run(
    ["bash", "-lc", f"source {shlex.quote(str(CANN_SET_ENV))} >/dev/null && env -0"],
    capture_output=True,
    check=True,
)
for entry in result.stdout.split(b"\0"):
    if b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[os.fsdecode(key)] = os.fsdecode(value)
os.environ["PY_BIN"] = PY_BIN
os.environ["PYTHON_BIN"] = PY_BIN
os.environ["ASCEND_SET_ENV"] = str(CANN_SET_ENV.resolve())

subprocess.run(
    [PY_BIN, "-m", "pip", "install", "pybind11", "expecttest"],
    check=True,
)

missing_tools = [name for name in ("cmake", "make", "c++", "bisheng") if shutil.which(name) is None]
if missing_tools:
    raise RuntimeError(f"缺少编译工具：{', '.join(missing_tools)}，请先安装并配置 PATH。")
cmake_version = subprocess.run(
    ["cmake", "--version"], capture_output=True, text=True, check=True
).stdout.splitlines()[0]
version = re.search(r"(\d+)\.(\d+)", cmake_version)
if version is None or tuple(map(int, version.groups())) < (3, 16):
    raise RuntimeError(f"需要 CMake 3.16 及以上，当前为：{cmake_version}")
python_header = Path(sysconfig.get_path("include")) / "Python.h"
if not python_header.is_file():
    raise FileNotFoundError(f"缺少当前 Python 环境的开发头文件：{python_header}")

# 使用继承 CANN 环境的独立进程检查 NPU，避免当前内核提前加载设备动态库。
subprocess.run(
    [PY_BIN, "-c", """
import torch
import torch_npu
import pybind11
import expecttest

assert torch.npu.is_available(), "NPU 不可用，请检查驱动、CANN 和 torch_npu 配置。"
x = torch.ones(1, device="npu")
y = x + x
torch.npu.synchronize()
assert y.cpu().item() == 2.0, "NPU 运算检查失败。"
print("torch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("NPU 可用数量:", torch.npu.device_count())
print("NPU 运算检查通过。")
"""],
    check=True,
)

exercise_files = {
    "softmax": [
        "common/softmax_custom.asc",
        "pybind/pybind_binding.cpp",
        "pybind/CMakeLists.txt",
        "pybind/softmax_test.py",
        "torch_library/torch_binding.cpp",
        "torch_library/CMakeLists.txt",
        "torch_library/softmax_test.py",
    ],
    "matmul": [
        "common/mmad_custom.asc",
        "pybind/pybind_binding.cpp",
        "pybind/CMakeLists.txt",
        "pybind/matmul_test.py",
        "torch_library/torch_binding.cpp",
        "torch_library/CMakeLists.txt",
        "torch_library/matmul_test.py",
    ],
}
created = 0
for operator, filenames in exercise_files.items():
    for filename in filenames:
        path = ROOT / operator / filename
        path.parent.mkdir(parents=True, exist_ok=True)
        try:
            with path.open("x", encoding="utf-8"):
                pass
            created += 1
            print(f"已创建：{path.relative_to(ROOT)}")
        except FileExistsError:
            if not path.is_file():
                raise RuntimeError(f"预期为文件，但该路径已被目录占用：{path}")
            print(f"已保留：{path.relative_to(ROOT)}")

print(f"CANN 环境脚本：{CANN_SET_ENV}")
print(f"编译架构：{ASC_ARCH}")
print(f"Python 解释器：{PY_BIN}")
print(f"编译工具：{cmake_version}")
print(f"练习目录：{ROOT}")
print(f"环境初始化完成，新建 {created} 个空文件，其余已有文件保持不变。")


---

## 练习一：Softmax算子

### 算子定义

实现二维张量沿最后一维的 Softmax，每一行独立进行归一化：

$$
y_{i,j}=\frac{\exp(x_{i,j})}{\sum_{k=0}^{127}\exp(x_{i,k})},\qquad 0\leq i,j<128.
$$

| 项目 | 规格 |
| --- | --- |
| 算子类型（OpType） | Softmax |
| 计算类型 | Vector 算子，MemBase 基础 API |
| 核函数名 | `softmax_custom` |
| 计算维度 | 最后一维，即 `dim=-1` |
| 输入输出设备 | 同一 NPU 设备 |
| 存储格式 | ND，连续存储 |

| 方向 | 名称 | Shape | PyTorch dtype | Ascend C 数据类型 |
| --- | --- | --- | --- | --- |
| 输入 | x | `(128, 128)` | `torch.float32` | `float` |
| 输出 | y | `(128, 128)` | `torch.float32` | `float` |

Python 调用接收一个输入张量并返回一个同形状、同类型的输出张量。输入保持不变，每行输出元素之和应在浮点误差范围内等于 1。测试脚本需对照 PyTorch 的 Softmax 运算检查输出数值、shape、dtype 和 device，并报告精度校验结果。

### 需要编写的文件

工程根目录：`Sources/03.08/softmax/`。以下文件由环境准备单元格创建。

| 文件 | 职责 |
| --- | --- |
| `common/softmax_custom.asc` | Softmax 算子及核函数启动入口 |
| `pybind/pybind_binding.cpp` | Pybind11 接入方式的 PyTorch Tensor 接口与 Python 模块绑定 |
| `pybind/CMakeLists.txt` | Pybind11 接入方式的完整编译配置 |
| `pybind/softmax_test.py` | Pybind11 接入方式的数据准备、算子调用和正确性校验 |
| `torch_library/torch_binding.cpp` | Torch Library 接入方式的 PyTorch Tensor 接口与算子注册 |
| `torch_library/CMakeLists.txt` | Torch Library 接入方式的完整编译配置 |
| `torch_library/softmax_test.py` | Torch Library 接入方式的数据准备、算子调用和正确性校验 |

`common/softmax_custom.asc` 为必需文件，另外完成所选接入方式目录下的全部文件。


---

## 练习二：Matmul算子

### 算子定义

实现无 Bias 的矩阵乘法，固定 $M=8192$、$K=8192$、$N=8192$。

本题的右矩阵使用转置后的连续存储约定：Python 接口接收左矩阵 $A\in\mathbb{R}^{M\times K}$ 和右矩阵的转置 $B_T\in\mathbb{R}^{N\times K}$，输出为：

$$
C=A B_T^{\mathsf T},\qquad C_{i,j}=\sum_{k=0}^{8191}A_{i,k}(B_T)_{j,k}.
$$

| 项目 | 规格 |
| --- | --- |
| 算子类型（OpType） | Matmul |
| 计算类型 | Cube 算子，Ascend C 矩阵计算基础 API |
| 核函数名 | `mmad_custom` |
| 目标产品 | Atlas A2/A3，编译架构 `dav-2201` |
| 左矩阵约定 | A 不转置 |
| 右矩阵约定 | 输入为转置后的 B_T，逻辑右矩阵为 B_T 的转置 |
| Bias | 无 |
| 输入输出设备 | 同一 NPU 设备 |
| 存储格式 | ND，连续存储 |

| 方向 | 名称 | Shape | PyTorch dtype | Ascend C 数据类型 |
| --- | --- | --- | --- | --- |
| 输入 | A | `(8192, 8192)`，即 `(M, K)` | `torch.float16` | `half` |
| 输入 | B_T | `(8192, 8192)`，即 `(N, K)` | `torch.float16` | `half` |
| 输出 | C | `(8192, 8192)`，即 `(M, N)` | `torch.float16` | `half` |

虽然本题 M、K、N 相等，右矩阵的转置约定仍属于接口语义；B_T 必须是按 `(N, K)` 排列的连续张量。

Python 调用接收两个输入张量并返回一个输出张量，输入保持不变。测试脚本需按照上述转置约定，与 PyTorch 矩阵乘法结果进行浮点精度比较，检查输出的 shape、dtype 和 device，并报告精度校验结果。

### 需要编写的文件

工程根目录：`Sources/03.08/matmul/`。以下文件由环境准备单元格创建。

| 文件 | 职责 |
| --- | --- |
| `common/mmad_custom.asc` | Matmul 算子及核函数启动入口，面向 A2/A3 |
| `pybind/pybind_binding.cpp` | Pybind11 接入方式的 PyTorch Tensor 接口与 Python 模块绑定 |
| `pybind/CMakeLists.txt` | Pybind11 接入方式的完整编译配置 |
| `pybind/matmul_test.py` | Pybind11 接入方式的数据准备、算子调用和正确性校验 |
| `torch_library/torch_binding.cpp` | Torch Library 接入方式的 PyTorch Tensor 接口与算子注册 |
| `torch_library/CMakeLists.txt` | Torch Library 接入方式的完整编译配置 |
| `torch_library/matmul_test.py` | Torch Library 接入方式的数据准备、算子调用和正确性校验 |

`common/mmad_custom.asc` 为必需文件，另外完成所选接入方式目录下的全部文件。


---

## 参考答案

完成练习后，可查看 [完整答案与运行说明](./answer/03.08/README.md)。两道题均提供 Pybind11 和 Torch Library 工程，以及独立的编译与正确性验证脚本。

在本 notebook 所在目录的终端中执行以下命令，可依次编译并验证全部答案：

```bash
bash answer/03.08/run_all.sh
```
